# XLZD Shell Theta MF-GP Workflow

This notebook assumes the shell-theta CNP stage has already been run and that the aggregated shell-theta CNP CSVs exist under `data/out/cnp`.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_mfgp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_mfgp"))

from mfgp_clean_pipeline import load_runtime_config, run_mfgp_transform_suite

CONFIG_PATH = REPO_ROOT / "xlzd_shell_theta" / "settings_shell_minibatch.yaml"
CNP_TRAIN_CSV = REPO_ROOT / "data" / "out" / "cnp" / "cnp_xlzd_shell_v1_minibatch_output_15epochs.csv"
CNP_VALIDATION_CSV = REPO_ROOT / "data" / "out" / "cnp" / "cnp_xlzd_shell_v1_minibatch_output_validation_15epochs.csv"
ITERATION = 0
GRID_POINTS = 120
PREDICT_CHUNK_SIZE = 20000
RANDOM_STATE = 42
TARGET_TRANSFORMS = ["linear", "log_hf", "log_lf", "log_both"]

print(f"Repo root: {REPO_ROOT}")
print(f"MF-GP config: {CONFIG_PATH}")
print(f"Training CNP CSV: {CNP_TRAIN_CSV}")
print(f"Validation CNP CSV: {CNP_VALIDATION_CSV}")
print(f"Target transforms: {TARGET_TRANSFORMS}")


## 1. Load And Inspect The Runtime Config

In [ ]:
runtime = load_runtime_config(CONFIG_PATH)

summary = pd.DataFrame(
    {
        "field": [
            "version",
            "theta_headers",
            "theta_min",
            "theta_max",
            "out_dir_cnp",
            "out_dir_mfgp",
        ],
        "value": [
            runtime.version,
            ", ".join(runtime.theta_headers),
            runtime.theta_min,
            runtime.theta_max,
            str(runtime.out_dir_cnp),
            str(runtime.out_dir_mfgp),
        ],
    }
)
summary


## 2. Fit The MF-GP Transform Suite

In [ ]:
mfgp_results = run_mfgp_transform_suite(
    config_path=CONFIG_PATH,
    cnp_csv=CNP_TRAIN_CSV,
    validation_csv=CNP_VALIDATION_CSV,
    transforms=TARGET_TRANSFORMS,
    iteration=ITERATION,
    grid_points_per_axis=GRID_POINTS,
    random_state=RANDOM_STATE,
    predict_chunk_size=PREDICT_CHUNK_SIZE,
    verbose=True,
)

EXPERIMENT_TITLES = {
    "linear": "Normal MF-GP",
    "log_hf": "Log HF: emulate log10(y_raw)",
    "log_lf": "Log LF: use log10(y_cnp)",
    "log_both": "Log HF + Log LF: use log10(y_raw) and log10(y_cnp)",
}

PLOT_SELECTIONS = {
    "linear": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_hf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_lf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_both": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
}

def _display_path_artifact(title: str, artifact_path: str) -> None:
    print(title)
    print(f"Open manually: {artifact_path}")

def _display_image_artifact(title: str, artifact_path: str) -> None:
    print(title)
    display(Image(filename=str(artifact_path)))

def display_selected_plots(mode: str) -> None:
    result = mfgp_results[mode]
    print(f"{EXPERIMENT_TITLES[mode]}\n")
    for title, attr_name in PLOT_SELECTIONS[mode]:
        artifact_path = getattr(result, attr_name, None)
        if not artifact_path:
            continue
        if str(artifact_path).lower().endswith(".html"):
            _display_path_artifact(title, str(artifact_path))
        else:
            _display_image_artifact(title, str(artifact_path))
        print()


## 3. Inspect Metrics And CSV Outputs

In [ ]:
metric_rows = []
for mode, result in mfgp_results.items():
    metrics = json.loads(Path(result.metrics_json).read_text())
    metric_rows.append({"mode": mode, **metrics})

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

for mode, result in mfgp_results.items():
    print(f"\n=== {mode} prediction CSV preview ===")
    display(pd.read_csv(result.prediction_csv).head())
    print(f"=== {mode} grid CSV preview ===")
    display(pd.read_csv(result.grid_csv).head())


## Normal MF-GP

In [ ]:
display_selected_plots("linear")


## Log HF

In [ ]:
display_selected_plots("log_hf")


## Log LF

In [ ]:
display_selected_plots("log_lf")


## Log HF + Log LF

In [ ]:
display_selected_plots("log_both")
